# MedCLIP Retrieval + Export Model

Bản này đã chỉnh lại **đường dẫn theo notebook Kaggle của nhóm**:

- Dataset ảnh: `simhadrisadaram/mimic-cxr-dataset`
- Dataset split CSV: `avcuongy/mimic-cxr-split`
- Root ảnh: `DATA_DIR/official_data_iccv_final`
- CSV: `mimic_train.csv`, `mimic_val.csv`, `mimic_test.csv`
- Cột ảnh: `best_image`
- Cột text: `text`

Pipeline:
1. Load pretrained MedCLIP-ResNet.
2. Đánh giá retrieval bằng R@1, R@5, R@10.
3. Fine-tune contrastive tối đa 10 epoch nếu cần.
4. Xuất full model, vision encoder, text encoder.


## 1. Install libraries


In [ ]:
!pip -q uninstall -y MedCLIP medclip transformers tokenizers

!pip -q install \
    "pandas==2.2.2" \
    "scikit-learn>=1.2,<1.9" \
    "pillow>=8.0,<12.0" \
    "tokenizers>=0.19,<0.20" \
    "transformers==4.41.2" \
    "huggingface_hub" \
    "safetensors" \
    "timm>=0.6.11" \
    "wget" \
    "textaugment>=1.3.4" \
    "kagglehub" \
    "tqdm"

!pip -q install --no-deps git+https://github.com/RyanWangZf/MedCLIP.git

# Kiểm tra nhanh sau khi cài
import pandas as pd
import sklearn
import PIL
import transformers
import tokenizers

print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("Pillow:", PIL.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)

from medclip import MedCLIPModel, MedCLIPVisionModel
print("MedCLIP import OK")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
pandas: 2.2.2
sklearn: 1.6.1
Pillow: 11.3.0
transformers: 4.41.2
tokenizers: 0.19.1
MedCLIP import OK


## 2. Import + config


In [ ]:
# ===== Import libraries =====
# Các thư viện chuẩn để xử lý file/path, dữ liệu bảng, ảnh, mô hình PyTorch và tokenizer.
import os
import re
import json
import math
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup

# ===== Reproducibility =====
# Cố định seed để kết quả train/evaluate ổn định hơn giữa các lần chạy.
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# Tự động dùng GPU nếu môi trường có CUDA.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ===== Main config =====
# MedCLIP là contrastive model nên số epoch, batch size và learning rate dùng cho fine-tune retrieval.
EPOCHS = 10
BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
MAX_TEXT_LEN = 128
IMG_SIZE = 224

# Set None để chạy full dataset.
# Nên để subset nhỏ trước để kiểm tra pipeline/path rồi mới train full.
TRAIN_SUBSET = None
VAL_SUBSET = None
TEST_SUBSET = None

# Kaggle mặc định cho phép ghi output vào /kaggle/working.
# Nếu chạy Colab, folder này vẫn được tạo thủ công để giữ cùng cấu trúc.
WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

# Tất cả model, metric và ví dụ retrieval sẽ được lưu ở đây.
OUTPUT_DIR = os.path.join(WORK_DIR, "medclip_retrieval_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Theo split CSV của nhóm:
# - best_image: path tương đối tới ảnh X-ray
# - text: report/caption tương ứng
IMAGE_COL = "best_image"
TEXT_COL = "text"

DEVICE: cuda


## 3. Download Kaggle datasets theo code của nhóm


In [ ]:
# ===== Download datasets from Kaggle =====
# Dataset 1: chứa ảnh MIMIC-CXR.
# Dataset 2: chứa các file split train/val/test mà nhóm đang dùng.

import kagglehub

DATA_DIR = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")
print("DATA_DIR:", DATA_DIR)

SPLIT_DIR = kagglehub.dataset_download("avcuongy/mimic-cxr-split")
print("SPLIT_DIR:", SPLIT_DIR)

# Ảnh thật nằm trong official_data_iccv_final.
ROOT_DIR = os.path.join(DATA_DIR, "official_data_iccv_final")

# Các file CSV split đã được nhóm chuẩn bị sẵn.
TRAIN_CSV = os.path.join(SPLIT_DIR, "mimic_train.csv")
VAL_CSV = os.path.join(SPLIT_DIR, "mimic_val.csv")
TEST_CSV = os.path.join(SPLIT_DIR, "mimic_test.csv")

print("ROOT_DIR:", ROOT_DIR)
print("TRAIN_CSV:", TRAIN_CSV)
print("VAL_CSV:", VAL_CSV)
print("TEST_CSV:", TEST_CSV)

# Dừng sớm nếu sai đường dẫn để tránh lỗi FileNotFound khó debug ở DataLoader.
assert os.path.exists(ROOT_DIR), f"Không thấy ROOT_DIR: {ROOT_DIR}"
assert os.path.exists(TRAIN_CSV), f"Không thấy TRAIN_CSV: {TRAIN_CSV}"
assert os.path.exists(VAL_CSV), f"Không thấy VAL_CSV: {VAL_CSV}"
assert os.path.exists(TEST_CSV), f"Không thấy TEST_CSV: {TEST_CSV}"

Using Colab cache for faster access to the 'mimic-cxr-dataset' dataset.
DATA_DIR: /kaggle/input/mimic-cxr-dataset
Using Colab cache for faster access to the 'mimic-cxr-split' dataset.
SPLIT_DIR: /kaggle/input/mimic-cxr-split
ROOT_DIR: /kaggle/input/mimic-cxr-dataset/official_data_iccv_final
TRAIN_CSV: /kaggle/input/mimic-cxr-split/mimic_train.csv
VAL_CSV: /kaggle/input/mimic-cxr-split/mimic_val.csv
TEST_CSV: /kaggle/input/mimic-cxr-split/mimic_test.csv


## 4. Load split CSV


In [ ]:
# ===== Load and validate split CSV =====

def clean_text(text):
    """
    Làm sạch report y tế:
    - chuyển về lowercase
    - bỏ nhãn 'findings:'
    - bỏ ký tự đặc biệt
    - chuẩn hóa khoảng trắng

    Text sau khi clean sẽ đưa vào ClinicalBERT tokenizer của MedCLIP.
    """
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.replace("findings:", "")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def resolve_image_path(p):
    p = str(p).strip()
    p = p.replace("\\", "/")

    candidates = [
        p,                                      # nếu CSV đã chứa absolute path
        os.path.join(ROOT_DIR, p),              # path chuẩn của dataset ảnh
        os.path.join(DATA_DIR, p),              # fallback nếu folder root khác
        os.path.join(ROOT_DIR, os.path.basename(p)),  # fallback theo filename
    ]

    for c in candidates:
        if os.path.exists(c):
            return c

    # Trả về path dự đoán để in debug rõ ràng nếu file không tồn tại.
    return os.path.join(ROOT_DIR, p)

def load_split(csv_path, subset=None, split_name="split"):
    """
    Đọc một file split CSV và trả về DataFrame chuẩn gồm:
        image_path | text

    Có kiểm tra exists_rate để phát hiện lỗi path ngay từ đầu.
    """
    df = pd.read_csv(csv_path)
    print(f"\n[{split_name}] Loaded:", csv_path, df.shape)
    print("Columns:", list(df.columns))

    assert IMAGE_COL in df.columns, f"Không có cột {IMAGE_COL}. Columns={list(df.columns)}"
    assert TEXT_COL in df.columns, f"Không có cột {TEXT_COL}. Columns={list(df.columns)}"

    # Chỉ giữ 2 cột cần cho MedCLIP retrieval.
    df = df[[IMAGE_COL, TEXT_COL]].copy()
    df.columns = ["image_path", "text"]

    # Clean text và bỏ sample rỗng.
    df["text"] = df["text"].apply(clean_text)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)

    # Chuyển path tương đối thành path đầy đủ tới ảnh.
    df["image_path"] = df["image_path"].apply(resolve_image_path)

    # Lấy subset để test nhanh pipeline.
    if subset is not None:
        df = df.sample(min(subset, len(df)), random_state=42).reset_index(drop=True)

    # Kiểm tra tất cả ảnh có tồn tại không.
    exists_rate = df["image_path"].apply(os.path.exists).mean()
    print(f"[{split_name}] exists_rate:", exists_rate)
    print(f"[{split_name}] sample path:", df.loc[0, "image_path"])
    print(f"[{split_name}] sample exists:", os.path.exists(df.loc[0, "image_path"]))
    print(f"[{split_name}] sample text:", df.loc[0, "text"][:200])

    if exists_rate < 1.0:
        missing = df[~df["image_path"].apply(os.path.exists)].head(5)
        print(f"[{split_name}] Missing examples:")
        display(missing)
        raise FileNotFoundError(
            "Có ảnh không tìm thấy. Kiểm tra ROOT_DIR, DATA_DIR, IMAGE_COL hoặc file split."
        )

    return df

# Load ba tập train/validation/test.
train_df = load_split(TRAIN_CSV, TRAIN_SUBSET, "train")
val_df = load_split(VAL_CSV, VAL_SUBSET, "val")
test_df = load_split(TEST_CSV, TEST_SUBSET, "test")

print("\nFinal sizes:")
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)


[train] Loaded: /kaggle/input/mimic-cxr-split/mimic_train.csv (50009, 5)
Columns: ['subject_id', 'view', 'best_image', 'path', 'text']
[train] exists_rate: 1.0
[train] sample path: /kaggle/input/mimic-cxr-dataset/official_data_iccv_final/files/p10/p10691024/s57871237/c5088e31-bddab153-3115e9c3-fc34b7b5-28f8bcb7.jpg
[train] sample exists: True
[train] sample text: low lung volumes are present this accentuates the size of the cardiac silhouette which is moderately enlarged the aorta is unfolded widening of the superior mediastinum is likely attributable to low l

[val] Loaded: /kaggle/input/mimic-cxr-split/mimic_val.csv (10000, 5)
Columns: ['subject_id', 'view', 'best_image', 'path', 'text']
[val] exists_rate: 1.0
[val] sample path: /kaggle/input/mimic-cxr-dataset/official_data_iccv_final/files/p15/p15514793/s50522856/1a5a4a2a-58332d18-3d881df3-b5ea93dd-6fbd6c40.jpg
[val] sample exists: True
[val] sample text: impression as compared to the previous radiograph the patient has undergone r

## 5. Dataset + transform


In [ ]:
# ===== Image transforms and Dataset class =====
# MedCLIP-ResNet nhận ảnh RGB 224x224.
# Ảnh X-ray gốc có thể là grayscale, nên khi đọc ảnh sẽ convert sang RGB.

try:
    from medclip import constants
    IMG_MEAN = constants.IMG_MEAN
    IMG_STD = constants.IMG_STD
    print("Use MedCLIP mean/std:", IMG_MEAN, IMG_STD)
except Exception:
    # Fallback nếu constants của package MedCLIP thay đổi hoặc import lỗi.
    IMG_MEAN = 0.485
    IMG_STD = 0.229
    print("Fallback mean/std:", IMG_MEAN, IMG_STD)

# Transform cho train có augmentation nhẹ để tăng khả năng tổng quát.
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.3),
    T.RandomRotation(degrees=5),
    T.ToTensor(),
    T.Normalize(mean=[IMG_MEAN]*3, std=[IMG_STD]*3),
])

# Validation/test không augmentation để đánh giá ổn định.
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[IMG_MEAN]*3, std=[IMG_STD]*3),
])

class MedCLIPRetrievalDataset(Dataset):
    """
    Dataset cho MedCLIP retrieval.

    Mỗi sample trả về:
    - pixel_values: tensor ảnh X-ray
    - input_ids: token ids của report
    - attention_mask: mask của tokenizer
    - text: report gốc để xuất ví dụ retrieval
    - image_path: path ảnh để debug/xuất kết quả
    """
    def __init__(self, df, tokenizer, transform, max_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Convert RGB vì pretrained MedCLIP ResNet dùng 3 kênh.
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)

        # Tokenize report bằng Bio_ClinicalBERT tokenizer.
        encoded = self.tokenizer(
            row["text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "pixel_values": image,
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "text": row["text"],
            "image_path": row["image_path"],
        }

Use MedCLIP mean/std: 0.5862785803043838 0.27950088968644304


## 6. Load pretrained MedCLIP safely


In [ ]:
# ===== Load pretrained MedCLIP safely =====
# MedCLIP gồm:
# - vision_model: ResNet image encoder
# - text_model: Bio_ClinicalBERT text encoder
#
# Không gọi model.from_pretrained() trực tiếp vì có thể lỗi key "position_ids"
# khi checkpoint cũ chạy với transformers mới. Hàm dưới load checkpoint thủ công
# và bỏ key không tương thích.

from medclip import MedCLIPModel, MedCLIPVisionModel, constants
from transformers import AutoTokenizer
import requests
import wget

TEXT_ENCODER_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER_NAME)

def load_medclip_resnet_safely(device):
    """
    Load MedCLIP-ResNet pretrained checkpoint.

    strict=False được dùng để tránh crash do khác version transformers.
    Trọng số chính của image/text encoder vẫn được load bình thường.
    """
    model = MedCLIPModel(vision_cls=MedCLIPVisionModel)

    pretrain_dir = "/content/pretrained/medclip-resnet"
    weight_path = os.path.join(pretrain_dir, constants.WEIGHTS_NAME)

    # Nếu máy chưa có checkpoint, tải zip pretrained từ URL chính thức của MedCLIP.
    if not os.path.exists(weight_path):
        os.makedirs(pretrain_dir, exist_ok=True)
        pretrained_url = requests.get(constants.PRETRAINED_URL_MEDCLIP_RESNET).text.strip()
        print("Download pretrained model from:", pretrained_url)
        zip_path = wget.download(pretrained_url, pretrain_dir)
        print("\nDownloaded:", zip_path)

        # Giải nén checkpoint vào pretrain_dir.
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(pretrain_dir)

    state_dict = torch.load(weight_path, map_location="cpu")

    # Fix mismatch với transformers mới:
    # checkpoint cũ có thể chứa key position_ids nhưng model mới không cần.
    for key in list(state_dict.keys()):
        if key.endswith("position_ids"):
            print("Remove unexpected key:", key)
            state_dict.pop(key)

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    print("Loaded:", weight_path)

    return model.to(device)

medclip_model = load_medclip_resnet_safely(DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 208MB/s]


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Download pretrained model from: https://storage.googleapis.com/pytrial/medclip-pretrained.zip

Downloaded: /content/pretrained/medclip-resnet/medclip-pretrained.zip
Remove unexpected key: text_model.model.embeddings.position_ids
Missing keys: []
Unexpected keys: []
Loaded: /content/pretrained/medclip-resnet/pytorch_model.bin


## 7. DataLoader


In [ ]:
# ===== Build DataLoaders =====
# DataLoader tạo batch cho train/evaluate.
# shuffle=True chỉ dùng cho train để batch đa dạng hơn.

train_ds = MedCLIPRetrievalDataset(train_df, tokenizer, train_transform, MAX_TEXT_LEN)
val_ds = MedCLIPRetrievalDataset(val_df, tokenizer, eval_transform, MAX_TEXT_LEN)
test_ds = MedCLIPRetrievalDataset(test_df, tokenizer, eval_transform, MAX_TEXT_LEN)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

## 8. Evaluate Recall@K


In [ ]:
# ===== Retrieval evaluation: Recall@K =====
# MedCLIP không sinh caption. Nó encode ảnh và text thành embedding,
# rồi đo similarity giữa hai embedding đó.

@torch.no_grad()
def encode_all(model, loader, device):
    """
    Encode toàn bộ ảnh và text trong một split.

    Output:
    - image_embeds: ma trận embedding ảnh [N, D]
    - text_embeds: ma trận embedding text [N, D]
    - all_paths/all_texts: thông tin gốc để xuất ví dụ retrieval
    """
    model.eval()
    image_embeds_list = []
    text_embeds_list = []
    all_texts = []
    all_paths = []

    for batch in tqdm(loader, desc="Encoding"):
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Encode ảnh và report bằng 2 encoder của MedCLIP.
        img_embeds = model.encode_image(pixel_values)
        txt_embeds = model.encode_text(input_ids=input_ids, attention_mask=attention_mask)

        image_embeds_list.append(img_embeds.cpu())
        text_embeds_list.append(txt_embeds.cpu())
        all_texts.extend(batch["text"])
        all_paths.extend(batch["image_path"])

    image_embeds = torch.cat(image_embeds_list, dim=0)
    text_embeds = torch.cat(text_embeds_list, dim=0)

    # Normalize để dot product tương đương cosine similarity.
    image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)
    text_embeds = text_embeds / text_embeds.norm(dim=1, keepdim=True)

    return image_embeds, text_embeds, all_paths, all_texts

def compute_recall_at_k(image_embeds, text_embeds, k_values=(1, 5, 10)):
    """
    Tính Recall@K cho retrieval.

    I2T_R@K:
        Với mỗi ảnh, report đúng có nằm trong top-K report gần nhất không?

    T2I_R@K:
        Với mỗi report, ảnh đúng có nằm trong top-K ảnh gần nhất không?
    """
    sims = image_embeds @ text_embeds.T
    labels = torch.arange(sims.size(0))

    metrics = {}

    # Image -> Text retrieval
    for k in k_values:
        k_eff = min(k, sims.size(1))
        topk = sims.topk(k_eff, dim=1).indices
        recall = (topk == labels.unsqueeze(1)).any(dim=1).float().mean().item()
        metrics[f"I2T_R@{k}"] = recall

    # Text -> Image retrieval
    sims_t = sims.T
    for k in k_values:
        k_eff = min(k, sims_t.size(1))
        topk = sims_t.topk(k_eff, dim=1).indices
        recall = (topk == labels.unsqueeze(1)).any(dim=1).float().mean().item()
        metrics[f"T2I_R@{k}"] = recall

    return metrics

@torch.no_grad()
def evaluate_retrieval(model, loader, device, split_name="val"):
    """
    Evaluate một split bằng R@1, R@5, R@10.
    Hàm này dùng cho cả pretrained baseline, validation sau từng epoch và test cuối.
    """
    image_embeds, text_embeds, paths, texts = encode_all(model, loader, device)
    metrics = compute_recall_at_k(image_embeds, text_embeds, k_values=(1, 5, 10))
    print(split_name, metrics)
    return metrics, image_embeds, text_embeds, paths, texts

# Baseline: đánh giá pretrained MedCLIP trước khi fine-tune.
baseline_val_metrics, _, _, _, _ = evaluate_retrieval(
    medclip_model,
    val_loader,
    DEVICE,
    "val_pretrained"
)

Encoding:   0%|          | 0/63 [00:00<?, ?it/s]

val_pretrained {'I2T_R@1': 0.0, 'I2T_R@5': 0.004000000189989805, 'I2T_R@10': 0.008999999612569809, 'T2I_R@1': 0.0, 'T2I_R@5': 0.003000000026077032, 'T2I_R@10': 0.008999999612569809}


## 9. Fine-tune contrastive 10 epoch


In [ ]:
# ===== Fine-tuning setup =====
# Nhóm chốt MedCLIP chủ yếu dùng để retrieval và export model.
# Nếu cần transfer lên MIMIC-CXR hiện tại, train tối đa 10 epoch với config cố định.

# Freeze text encoder để train nhẹ và giảm overfitting.
# Lý do: Bio_ClinicalBERT đã học tốt ngôn ngữ y tế, nên chỉ cần chỉnh nhẹ projection/image side.
for name, p in medclip_model.text_model.named_parameters():
    p.requires_grad = False

# Mở projection head của text encoder nếu tồn tại để alignment linh hoạt hơn.
if hasattr(medclip_model.text_model, "projection_head"):
    for name, p in medclip_model.text_model.projection_head.named_parameters():
        p.requires_grad = True

# MedCLIP đã có built-in contrastive loss khi gọi return_loss=True.
criterion = "MedCLIP built-in contrastive loss"

# AdamW thường ổn định cho fine-tune pretrained model.
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, medclip_model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# Cosine schedule có warmup giúp learning rate tăng nhẹ lúc đầu rồi giảm dần.
total_steps = max(1, len(train_loader) * EPOCHS)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * total_steps)),
    num_training_steps=total_steps
)

print("Criterion:", criterion)
print("Optimizer:", optimizer)
print("Scheduler:", scheduler)
print("Trainable params:", sum(p.numel() for p in medclip_model.parameters() if p.requires_grad))

Criterion: MedCLIP built-in contrastive loss
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 2e-05
    lr: 0.0
    maximize: False
    weight_decay: 0.0001
)
Scheduler: <torch.optim.lr_scheduler.LambdaLR object at 0x7c9c4bc17ec0>
Trainable params: 24949825


In [ ]:
# ===== Train loop =====

def train_one_epoch(model, loader, optimizer, scheduler, device):
    """
    Train MedCLIP trong 1 epoch bằng contrastive loss.

    Mỗi batch gồm N cặp ảnh-report đúng.
    Model học để embedding ảnh i gần với report i,
    và xa với các report khác trong cùng batch.
    """
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="Train"):
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()

        # return_loss=True để MedCLIP tự tính contrastive loss ảnh-text.
        outputs = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_loss=True,
        )

        loss = outputs["loss_value"]
        loss.backward()

        # Gradient clipping tránh gradient quá lớn khi fine-tune.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / max(1, len(loader))

history = []
best_r1 = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(medclip_model, train_loader, optimizer, scheduler, DEVICE)

    # Dùng validation I2T_R@1 để chọn checkpoint tốt nhất.
    val_metrics, _, _, _, _ = evaluate_retrieval(
        medclip_model,
        val_loader,
        DEVICE,
        f"val_epoch_{epoch}"
    )

    row = {"epoch": epoch, "train_loss": train_loss}
    row.update(val_metrics)
    history.append(row)

    print(f"Epoch {epoch}/{EPOCHS} | loss={train_loss:.4f} | I2T_R@1={val_metrics['I2T_R@1']:.4f}")

    # Lưu checkpoint tốt nhất theo khả năng image-to-text retrieval.
    if val_metrics["I2T_R@1"] > best_r1:
        best_r1 = val_metrics["I2T_R@1"]
        save_path = os.path.join(OUTPUT_DIR, "best_medclip_retrieval.pt")
        torch.save(medclip_model.state_dict(), save_path)
        print("Saved best:", save_path)

# Lưu lịch sử train để đưa vào báo cáo hoặc debug.
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(OUTPUT_DIR, "train_history.csv"), index=False)
display(history_df)

Train:   0%|          | 0/625 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c9ce420fd80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c9ce420fd80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Encoding:   0%|          | 0/63 [00:00<?, ?it/s]

val_epoch_1 {'I2T_R@1': 0.00800000037997961, 'I2T_R@5': 0.023000000044703484, 'I2T_R@10': 0.04500000178813934, 'T2I_R@1': 0.004999999888241291, 'T2I_R@5': 0.028999999165534973, 'T2I_R@10': 0.04800000041723251}
Epoch 1/10 | loss=2.5424 | I2T_R@1=0.0080
Saved best: /kaggle/working/medclip_retrieval_outputs/best_medclip_retrieval.pt


Train:   0%|          | 0/625 [00:00<?, ?it/s]

...

Train:   0%|          | 0/625 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c9ce420fd80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c9ce420fd80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Encoding:   0%|          | 0/63 [00:00<?, ?it/s]

val_epoch_10 {'I2T_R@1': 0.006000000052154064, 'I2T_R@5': 0.03099999949336052, 'I2T_R@10': 0.06700000166893005, 'T2I_R@1': 0.014999999664723873, 'T2I_R@5': 0.0430000014603138, 'T2I_R@10': 0.0689999982714653}
Epoch 10/10 | loss=1.5228 | I2T_R@1=0.0060


,epoch,train_loss,I2T_R@1,I2T_R@5,I2T_R@10,T2I_R@1,T2I_R@5,T2I_R@10
0,1,2.542364,0.008,0.023,0.045,0.005,0.029,0.048
1,2,2.242478,0.008,0.027,0.051,0.012,0.043,0.063
2,3,2.103226,0.006,0.026,0.056,0.012,0.034,0.060
3,4,1.991701,0.005,0.030,0.061,0.013,0.037,0.065
4,5,1.858212,0.006,0.037,0.072,0.014,0.039,0.069
5,6,1.755270,0.010,0.036,0.069,0.017,0.046,0.077
6,7,1.661899,0.005,0.024,0.068,0.015,0.038,0.068
7,8,1.564793,0.007,0.027,0.066,0.015,0.039,0.068
8,9,1.528972,0.005,0.032,0.059,0.015,0.043,0.070
9,10,1.522845,0.006,0.031,0.067,0.015,0.043,0.069


## 10. Test evaluation + retrieval examples


In [ ]:
# ===== Test evaluation and retrieval examples =====
# Đây là phần "suy ra" của MedCLIP:
# với mỗi ảnh query, model tìm report gần nhất trong tập test.

best_path = os.path.join(OUTPUT_DIR, "best_medclip_retrieval.pt")
if os.path.exists(best_path):
    state = torch.load(best_path, map_location=DEVICE)
    medclip_model.load_state_dict(state, strict=False)
    print("Loaded best model:", best_path)

# Evaluate cuối cùng trên test set.
test_metrics, test_img_embeds, test_txt_embeds, test_paths, test_texts = evaluate_retrieval(
    medclip_model,
    test_loader,
    DEVICE,
    "test"
)

# Lưu metric R@K.
metrics_df = pd.DataFrame([test_metrics])
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "test_retrieval_metrics.csv"), index=False)
display(metrics_df)

# Xuất một số ví dụ image-to-text retrieval top-5 để xem model truy xuất report nào.
sims = test_img_embeds @ test_txt_embeds.T
examples = []

for i in range(min(20, sims.size(0))):
    topk = sims[i].topk(min(5, sims.size(1))).indices.tolist()

    examples.append({
        "query_image": test_paths[i],
        "ground_truth_text": test_texts[i],
        "top1_text": test_texts[topk[0]],
        "top1_is_correct": int(topk[0] == i),
        "top5_indices": topk,
    })

examples_df = pd.DataFrame(examples)
examples_df.to_csv(os.path.join(OUTPUT_DIR, "retrieval_examples.csv"), index=False)
display(examples_df.head())

Loaded best model: /kaggle/working/medclip_retrieval_outputs/best_medclip_retrieval.pt


Encoding:   0%|          | 0/63 [00:00<?, ?it/s]

test {'I2T_R@1': 0.004000000189989805, 'I2T_R@5': 0.026000000536441803, 'I2T_R@10': 0.06300000101327896, 'T2I_R@1': 0.012000000104308128, 'T2I_R@5': 0.03700000047683716, 'T2I_R@10': 0.06800000369548798}


,I2T_R@1,I2T_R@5,I2T_R@10,T2I_R@1,T2I_R@5,T2I_R@10
0,0.004,0.026,0.063,0.012,0.037,0.068


,query_image,ground_truth_text,top1_text,top1_is_correct,top5_indices
0,/kaggle/input/mimic-cxr-dataset/official_data_...,impression in comparison with the study of the...,as compared to the previous radiograph there i...,0,"[524, 895, 953, 650, 494]"
1,/kaggle/input/mimic-cxr-dataset/official_data_...,following removal of a right sided chest tube ...,impression in comparison with the study of cys...,0,"[697, 453, 806, 210, 492]"
2,/kaggle/input/mimic-cxr-dataset/official_data_...,impression new nasogastric tube ends in the up...,cardiomediastinal silhouette is unremarkable s...,0,"[644, 988, 966, 99, 167]"
3,/kaggle/input/mimic-cxr-dataset/official_data_...,impression no previous images relatively low l...,the heart is normal in size the mediastinal an...,0,"[754, 393, 174, 616, 60]"
4,/kaggle/input/mimic-cxr-dataset/official_data_...,ap upright and lateral views of the chest are ...,cardiomediastinal silhouette is within normal ...,0,"[213, 697, 375, 674, 675]"


## 11. Export model


In [ ]:
# ===== Export model checkpoints =====
# Sau khi train/evaluate, xuất ra 3 loại checkpoint để nhóm dùng tiếp.

# 1. Full MedCLIP checkpoint:
# Dùng khi muốn chạy lại retrieval đầy đủ image encoder + text encoder.
full_model_path = os.path.join(OUTPUT_DIR, "medclip_full_final.pt")
torch.save(medclip_model.state_dict(), full_model_path)

# 2. Vision encoder:
# File quan trọng nhất nếu nhóm muốn lấy encoder ảnh để gắn với LSTM/Transformer decoder.
vision_encoder_path = os.path.join(OUTPUT_DIR, "medclip_vision_encoder_final.pt")
torch.save(medclip_model.vision_model.state_dict(), vision_encoder_path)

# 3. Text encoder:
# Dùng khi cần encode report hoặc tính similarity ảnh-text về sau.
text_encoder_path = os.path.join(OUTPUT_DIR, "medclip_text_encoder_final.pt")
torch.save(medclip_model.text_model.state_dict(), text_encoder_path)

# 4. Metadata:
# Lưu thông tin config và metric test để người khác biết model được train/evaluate như thế nào.
meta = {
    "model": "MedCLIP-ResNet",
    "text_encoder": TEXT_ENCODER_NAME,
    "image_encoder": "ResNet50 via MedCLIPVisionModel",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "train_subset": TRAIN_SUBSET,
    "val_subset": VAL_SUBSET,
    "test_subset": TEST_SUBSET,
    "test_metrics": test_metrics,
}

meta_path = os.path.join(OUTPUT_DIR, "medclip_export_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved files:")
print(full_model_path)
print(vision_encoder_path)
print(text_encoder_path)
print(meta_path)

Saved files:
/kaggle/working/medclip_retrieval_outputs/medclip_full_final.pt
/kaggle/working/medclip_retrieval_outputs/medclip_vision_encoder_final.pt
/kaggle/working/medclip_retrieval_outputs/medclip_text_encoder_final.pt
/kaggle/working/medclip_retrieval_outputs/medclip_export_metadata.json


## 12. Zip outputs để tải về hoặc lưu Drive


In [ ]:
# ===== Zip all outputs =====
# Bên trong gồm model checkpoints, metrics, retrieval examples và metadata.

import shutil

zip_base = os.path.join(WORK_DIR, "medclip_retrieval_outputs")
zip_path = zip_base + ".zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(zip_base, "zip", OUTPUT_DIR)

print("ZIP:", zip_path)
print("Output folder:", OUTPUT_DIR)

ZIP: /kaggle/working/medclip_retrieval_outputs.zip
Output folder: /kaggle/working/medclip_retrieval_outputs


In [ ]:
from google.colab import files
files.download("/kaggle/working/medclip_retrieval_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>